<a href="https://colab.research.google.com/github/ERA-Software/computational-data-analysis/blob/main/notebooks/S6_classification_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification Demo: Logistic Regression on Bike Sharing Data

**Lecture 6: From predicted probabilities to classification**

This notebook accompanies the classification lecture. We use the UCI Bike Sharing dataset and derive a binary **surge** label (demand > 95th percentile) to demonstrate:

1. Data preparation and surge label
2. Logistic regression basics (sigmoid, log-odds, cross-entropy)
3. Fitting and predicted probabilities
4. Threshold effects on predictions
5. Confusion matrices and metric trade-offs
6. ROC and Precision-Recall curves
7. Calibration diagnostics
8. Imbalance handling strategies (class weights, SMOTE)
9. Cost-sensitive threshold tuning
10. Bonus: Tree-based model predictions

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, balanced_accuracy_score,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    log_loss, ConfusionMatrixDisplay
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Plotting defaults
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'figure.facecolor': 'white',
})

# Output directory for figures
# FIG_DIR = Path("figures")
# FIG_DIR.mkdir(exist_ok=True)

## 1. Data Preparation

We load the UCI Bike Sharing (hourly) dataset and engineer a binary **surge** label: $y = 1$ if hourly demand (`cnt`) exceeds its 95th percentile.

In [ ]:
# Load the Bike Sharing dataset from a local CSV if available,
# otherwise simulate realistic data matching the UCI structure.

_bike_csv = "hour.csv"  # Place UCI hour.csv here if available
if os.path.exists(_bike_csv):
    df = pd.read_csv(_bike_csv)
    print(f"Loaded dataset from {df.shape[0]} rows and {df.shape[1]} columns.")
else:
    # Simulate a realistic hourly bike-sharing dataset (17379 rows)
    np.random.seed(42)
    n_obs = 17379
    # Generate sequential hours across ~2 years
    total_hours = np.arange(n_obs)
    hr = total_hours % 24
    day_of_year = (total_hours // 24) % 365
    mnth = np.clip((day_of_year // 30) + 1, 1, 12)
    season = np.where(mnth <= 3, 1, np.where(mnth <= 6, 2, np.where(mnth <= 9, 3, 4)))
    yr = (total_hours // (365 * 24)).astype(int)
    weekday = (total_hours // 24) % 7
    workingday = ((weekday >= 1) & (weekday <= 5)).astype(int)
    holiday = (np.random.rand(n_obs) < 0.03).astype(int)
    weathersit = np.random.choice([1, 2, 3, 4], n_obs, p=[0.6, 0.25, 0.12, 0.03])
    temp = 0.5 + 0.3 * np.sin(2 * np.pi * (mnth - 1) / 12) + 0.05 * np.random.randn(n_obs)
    temp = np.clip(temp, 0, 1)
    atemp = temp + 0.02 * np.random.randn(n_obs)
    atemp = np.clip(atemp, 0, 1)
    hum = 0.6 + 0.15 * np.random.randn(n_obs)
    hum = np.clip(hum, 0, 1)
    windspeed = np.clip(0.2 + 0.1 * np.random.randn(n_obs), 0, 0.7)

    # Demand model: strong feature-driven signal for pedagogical clarity
    # Hourly pattern: double peak (morning + evening rush)
    hour_effect = (
        80 * np.exp(-0.5 * ((hr - 8) / 1.5)**2) +    # morning rush
        120 * np.exp(-0.5 * ((hr - 17) / 2.0)**2) +   # evening rush
        40 * np.exp(-0.5 * ((hr - 12) / 1.5)**2)      # lunch bump
    )
    # Seasonal effect: summer peak
    season_effect = 100 * np.sin(np.pi * (mnth - 1) / 11)
    # Year-over-year growth
    year_effect = 80 * yr
    # Weather: strong penalty for bad weather
    weather_effect = np.where(weathersit == 1, 30,
                     np.where(weathersit == 2, 0,
                     np.where(weathersit == 3, -80, -160)))
    # Working day interaction with rush hours
    workday_rush = 60 * workingday * (((hr >= 7) & (hr <= 9)) | ((hr >= 16) & (hr <= 19))).astype(float)
    # Temperature effect (nonlinear)
    temp_effect = 120 * temp - 40 * (temp - 0.5)**2
    # Combine
    base = 80 + hour_effect + season_effect + year_effect + weather_effect.astype(float) + workday_rush + temp_effect
    noise = np.random.exponential(25, n_obs)
    cnt = np.maximum(1, (base + noise)).astype(int)

    df = pd.DataFrame({
        'season': season, 'yr': yr, 'mnth': mnth, 'hr': hr,
        'holiday': holiday, 'weekday': weekday, 'workingday': workingday,
        'weathersit': weathersit, 'temp': temp, 'atemp': atemp,
        'hum': hum, 'windspeed': windspeed, 'cnt': cnt
    })

print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Define the surge label
threshold_95 = df['cnt'].quantile(0.95)
df['surge'] = (df['cnt'] >= threshold_95).astype(int)
print(f"95th percentile of demand: {threshold_95:.0f}")
print(f"Surge prevalence: {df['surge'].mean():.3f} ({df['surge'].sum()} / {len(df)})")

In [ ]:
# Select features for classification
feature_cols = ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday',
                'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed']

X = df[feature_cols].values
y = df['surge'].values

print(f"Features: {len(feature_cols)}")
print(f"Class distribution: 0={np.sum(y==0)}, 1={np.sum(y==1)}")

### Feature Description: Bike Sharing Dataset

| Feature | Description |
|---|---|
| `season` | Season (1=winter, 2=spring, 3=summer, 4=fall) |
| `yr` | Year (0=2011, 1=2012); captures overall ridership growth trend |
| `mnth` | Month (1--12) |
| `hr` | Hour of day (0--23); the strongest predictor of surge? |
| `holiday` | Binary: 1 if public holiday |
| `weekday` | Day of the week (0--6) |
| `workingday` | Binary: 1 if neither weekend nor holiday |
| `weathersit` | Weather category (1=clear/partly cloudy, 2=mist/cloudy, 3=light rain or snow, 4=heavy rain/ice/fog) |
| `temp` | Normalized temperature in Celsius |
| `atemp` | Normalized feels-like temperature in Celsius |
| `hum` | Normalized humidity: raw value divided by 100 |
| `windspeed` | Normalized wind speed: raw value divided by 67 |

## 2. Logistic Regression: The Basics

Before fitting a model, let us review the core ideas behind logistic regression.

**Why not linear regression for classification?**
A linear model $\hat{y} = \mathbf{x}^\top \boldsymbol{\beta}$ can produce values outside $[0, 1]$, which makes no sense as a probability. Logistic regression fixes this by passing the linear combination through the **sigmoid** (logistic) function:

$$\hat{p}(x) = \sigma(z) = \frac{1}{1 + e^{-z}}, \quad z = \mathbf{x}^\top \boldsymbol{\beta}$$

**Key properties of the sigmoid:**

- Output is always in $(0, 1)$, so it can be interpreted as a probability.
- At $z = 0$, $\sigma(0) = 0.5$ (the decision boundary when $\tau = 0.5$).
- It is monotonic: larger $z$ means higher predicted probability.

**Log-odds (logit) interpretation:**
The model is linear in the log-odds:

$$\log \frac{\hat{p}}{1 - \hat{p}} = \mathbf{x}^\top \boldsymbol{\beta}$$

Each coefficient $\beta_j$ tells you how much a one-unit increase in feature $j$ changes the log-odds of the positive class. Exponentiate to get the odds ratio: $e^{\beta_j}$.

**How is it trained?**
Unlike linear regression (which minimizes squared error), logistic regression maximizes the **log-likelihood** -- equivalently, it minimizes the **cross-entropy loss**:

$$\mathscr{L} = -\frac{1}{n}\sum_{i=1}^{n}\bigl[y_i \log \hat{p}_i + (1 - y_i)\log(1 - \hat{p}_i)\bigr]$$

There is no closed-form solution; the optimizer uses iterative methods (e.g., L-BFGS).

In [ ]:
# Figure: The sigmoid function and the effect of different coefficient magnitudes
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

z = np.linspace(-8, 8, 300)

# Left panel: the sigmoid curve
ax = axes[0]
sigmoid = 1 / (1 + np.exp(-z))
ax.plot(z, sigmoid, color='royalblue', lw=2.5)
ax.axhline(0.5, color='gray', linestyle='--', lw=1, alpha=0.6)
ax.axvline(0.0, color='gray', linestyle='--', lw=1, alpha=0.6)
ax.set_xlabel('$z = \\mathbf{x}^\\top \\boldsymbol{\\beta}$')
ax.set_ylabel('$\\sigma(z)$')
ax.set_title('The Sigmoid Function')
ax.annotate('$\\sigma(0) = 0.5$', xy=(0, 0.5), xytext=(2.5, 0.35),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='firebrick'),
            color='firebrick')
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.3)

# Right panel: effect of coefficient magnitude on steepness
ax = axes[1]
for beta, color, ls in [(0.5, 'forestgreen', '--'), (1.0, 'royalblue', '-'),
                         (3.0, 'firebrick', '-.')]:
    sig = 1 / (1 + np.exp(-beta * z))
    ax.plot(z, sig, color=color, lw=2, linestyle=ls, label=f'$\\beta = {beta}$')
ax.set_xlabel('Feature value $x$')
ax.set_ylabel('$\\hat{p}(x)$')
ax.set_title('Coefficient Magnitude Controls Steepness')
ax.legend()
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.3)

plt.tight_layout()
# plt.savefig(FIG_DIR / 'fig_sigmoid.png', bbox_inches='tight')
plt.show()

The left panel shows the standard sigmoid curve. The right panel illustrates how a larger coefficient $\beta$ makes the transition from 0 to 1 sharper, meaning the model is more "confident" in its predictions near the boundary.

## 3. Fitting and Predicted Scores

Now we fit a logistic regression to the surge data and examine the predicted probabilities (scores).

In [ ]:
# Stratified train/test split (80/20) for clean pedagogical illustrations.
# In practice you would use a time-based split; we use stratified here so
# the figures show clear metric behaviour without distribution shift.
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(y_train)} (surge={y_train.sum()})")
print(f"Test:  {len(y_test)} (surge={y_test.sum()})")

## Train/Test Split Strategy: Stratified vs. Time-Based

The comment in the code flags an important methodological choice. Here is what each strategy does and why the distinction matters.

---

### Stratified Split (used here)

`stratify=y` instructs scikit-learn to preserve the class proportion of `y` in both the train and test sets. If 15% of samples are surge events in the full dataset, both splits will be approximately 15% surge.

Since the goal of this notebook is to illustrate how a classifier behaves on metrics like precision, recall, or AUC, a stratified split removes *distribution shift* as a confounding variable. Both sets are drawn i.i.d. from the same underlying distribution, so metric curves are stable and interpretable.

**The critical assumption it violates:** samples are treated as exchangeable -- the model is allowed to train on data from, say, Tuesday and test on Monday. For temporal data, this is **data leakage**: future information implicitly informs the model about the past.

---

### Time-Based Split (the correct production approach)

A time-based split respects the arrow of time. A common implementation:

```python
cutoff = int(len(X) * 0.8)
X_train, X_test = X.iloc[:cutoff], X.iloc[cutoff:]
y_train, y_test = y.iloc[:cutoff], y.iloc[cutoff:]
```

The model trains on the first 80% of chronological observations and tests on the last 20%. This faithfully simulates deployment: you always predict forward in time, never backward.

**The tradeoff:** The class balance in the test set is whatever the data naturally produces in that period. If surge events cluster seasonally, the test set may be far more (or less) surge-heavy than the training set, which is *realistic* but can make metric comparisons across experiments noisy.

---

### Summary Table

| Property | Stratified | Time-Based |
|---|---|---|
| Class balance preserved | Yes, by construction | No, reflects natural drift |
| Data leakage | Yes (temporal) | No |
| Metric stability | High | Lower (distribution shift) |
| Production validity | Low | High |
| Appropriate use | Pedagogy, cross-sectional data | Time series, forecasting, deployment evaluation |

---

> **Bottom line:** Use stratified splitting to produce clean figures that isolate classifier behavior from distributional noise. Use time-based splitting whenever the model will be deployed to predict future observations -- which is almost always the case for operational forecasting tasks.

In [ ]:
# Fit logistic regression
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])
pipe_lr.fit(X_train, y_train)

# Predicted probabilities on test set
probs_lr = pipe_lr.predict_proba(X_test)[:, 1]

In [ ]:
# Figure 1: Distribution of predicted probabilities
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(probs_lr[y_test == 0], bins=50, alpha=0.6, label='Actual Negative (y=0)', color='steelblue', density=True)
ax.hist(probs_lr[y_test == 1], bins=50, alpha=0.6, label='Actual Positive (y=1)', color='firebrick', density=True)
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label=r'Default $\tau=0.5$')
ax.set_xlabel('Predicted probability $\hat{p}(x)$')
ax.set_ylabel('Density')
ax.set_title('Distribution of Predicted Probabilities (Logistic Regression)')
ax.legend()
ax.set_xlim(0, 1)
plt.tight_layout()
# plt.savefig(FIG_DIR / 'fig_score_distribution.png', bbox_inches='tight')
plt.show()

- Severe class imbalance is confirmed visually. The blue mass (y=0) is almost entirely concentrated at $\hat{p}(x) < 0.05$, with very high density. The model correctly assigns low probabilities to most negatives.

- The model is poorly calibrated for positives. The red distribution (y=1) is spread diffusely across $[0, 0.35]$ with no clear mode away from zero. The model is uncertain about positives and never confidently predicts them -- no red mass appears near or above $\tau = 0.5$.

- The default threshold $\tau = 0.5$ is useless here. Because no predicted probability (for either class) exceeds ~0.55, the dashed line classifies virtually everything as negative. This will produce near-zero recall on the positive class despite potentially high accuracy -- a classic imbalance pitfall.

**What to do:**

- **Lower the threshold** (e.g. $\tau \in [0.1, 0.2]$) to recover positive-class recall. Use a precision-recall curve to choose $\tau$ based on your cost tradeoff.
- **Check calibration**: logistic regression with imbalanced data often underestimates $P(y=1)$; consider `class_weight='balanced'` or Platt scaling.
- **Consider resampling or reweighting** if the imbalance ratio is extreme (which the blue spike suggests it may be).

In [ ]:
# Examine logistic regression coefficients
feature_names = feature_cols
coefficients = pipe_lr.named_steps['lr'].coef_[0]
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients,
    'abs_coefficient': np.abs(coefficients)
}).sort_values(by='abs_coefficient', ascending=False)
print(coef_df)

## Inspecting Logistic Regression Coefficients

After fitting the pipeline, this block extracts and ranks the model's learned coefficients to understand which features drive predictions most strongly.

```python
feature_names = feature_cols          # the original list of input feature names
coefficients = pipe_lr.named_steps['lr'].coef_[0]  # shape (n_features,)
```

`named_steps['lr']` navigates into the pipeline to retrieve the fitted `LogisticRegression` object. `.coef_` has shape `(1, n_features)` for binary classification -- the `[0]` unpacks it to a 1-D array.

Each coefficient $\beta_j$ represents the change in **log-odds** of the positive class per one-unit increase in feature $j$ (after any preprocessing applied earlier in the pipeline, e.g. scaling):

$$\log \frac{P(y=1 \mid \mathbf{x})}{P(y=0 \mid \mathbf{x})} = \beta_0 + \sum_j \beta_j x_j$$

The DataFrame is then sorted by `abs_coefficient` (descending), so the most influential features appear first regardless of direction:

- A **large positive** $\beta_j$: feature pushes toward the positive class (surge).
- A **large negative** $\beta_j$: feature pushes toward the negative class.
- A coefficient near **zero**: the model found the feature uninformative (or it was regularized away).

> **Interpretation caveat:** coefficients are only directly comparable if features were standardized beforehand (e.g. via `StandardScaler` in the pipeline). On raw, differently-scaled features, magnitude comparisons are misleading.

## 4. Threshold Effects

The same model produces very different predictions depending on the threshold $\tau$.

In [ ]:
# Figure 2: Threshold effects -- predicted positives and metric trade-offs
thresholds = [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]

results = []
for tau in thresholds:
    y_pred = (probs_lr >= tau).astype(int)
    results.append({
        'Threshold': tau,
        'Pred Pos': y_pred.sum(),
        'TP': int(np.sum((y_pred == 1) & (y_test == 1))),
        'FP': int(np.sum((y_pred == 1) & (y_test == 0))),
        'FN': int(np.sum((y_pred == 0) & (y_test == 1))),
        'TN': int(np.sum((y_pred == 0) & (y_test == 0))),
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_test, y_pred),
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

Threshold Sensitivity Analysis

| Metric | What it measures |
|---|---|
| **Precision** | Of all predicted positives, what fraction are truly positive. High precision = few false alarms. |
| **Recall** | Of all actual positives, what fraction are caught. High recall = few missed surges. |
| **F1** | Harmonic mean of precision and recall. Balances the two; useful when classes are imbalanced. |
| **MCC** | Matthews Correlation Coefficient. Accounts for all four cells of the confusion matrix. Ranges from -1 to 1; more reliable than F1 under heavy imbalance. |
| **Accuracy** | Fraction of correct predictions overall. **Misleading here**: predicting all-negative ($\tau \geq 0.5$) scores 95% accuracy while catching zero surges. |

---

### What the table shows

At $\tau = 0.5$ and above, the model predicts no positives at all -- recall and F1 collapse to zero.

The best tradeoff is at **$\tau = 0.1$**: MCC = 0.32, recall = 0.68, but precision is only 0.21, meaning roughly 4 in 5 flagged events are false positives.

At **$\tau = 0.2$**: precision improves to 0.29 and F1 is nearly identical (0.322 vs 0.319), but recall drops to 0.36 -- you miss nearly two-thirds of actual surges.

> Lowering $\tau$ recovers surges (recall) at the cost of more false alarms (precision). Which threshold to prefer depends on the relative cost of a missed surge vs. a false alert in your application.

In [ ]:
# Figure 2: Precision-Recall-F1 vs threshold
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: metrics vs threshold
ax = axes[0]
ax.plot(df_results['Threshold'], df_results['Precision'], 'o-', label='Precision', color='darkorange')
ax.plot(df_results['Threshold'], df_results['Recall'], 's-', label='Recall', color='forestgreen')
ax.plot(df_results['Threshold'], df_results['F1'], 'D-', label='F1', color='royalblue')
ax.plot(df_results['Threshold'], df_results['MCC'], '^-', label='MCC', color='purple')
ax.set_xlabel(r'Threshold $\tau$')
ax.set_ylabel('Score')
ax.set_title('Metrics vs. Decision Threshold')
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)

# Right: predicted positives vs threshold
ax = axes[1]
ax.bar(df_results['Threshold'].astype(str), df_results['Pred Pos'], color='steelblue', alpha=0.7)
ax.axhline(y_test.sum(), color='firebrick', linestyle='--', label=f'Actual positives ({y_test.sum()})')
ax.set_xlabel(r'Threshold $\tau$')
ax.set_ylabel('Number of predicted positives')
ax.set_title('Predicted Positives at Each Threshold')
ax.legend()
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
# plt.savefig(FIG_DIR / 'fig_threshold_effects.png', bbox_inches='tight')
plt.show()

## 5. Confusion Matrix

We visualize the confusion matrix at two different thresholds to show the trade-off.

In [ ]:
# Figure 3: Confusion matrices at two thresholds
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for i, tau in enumerate([0.3, 0.5]):
    y_pred = (probs_lr >= tau).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Surge', 'Surge'])
    disp.plot(ax=axes[i], cmap='Blues', colorbar=False)
    axes[i].set_title(f'Confusion Matrix ($\\tau = {tau}$)')

plt.tight_layout()
# plt.savefig(FIG_DIR / 'fig_confusion_matrices.png', bbox_inches='tight')
plt.show()

## 6. ROC and Precision-Recall Curves

These curves evaluate the classifier across all possible thresholds simultaneously.

In [ ]:
# Figure 4: ROC and PR curves side by side
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# ROC
fpr, tpr, _ = roc_curve(y_test, probs_lr)
roc_auc = auc(fpr, tpr)
axes[0].plot(fpr, tpr, color='royalblue', lw=2, label=f'Logistic Reg. (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='royalblue')
axes[0].set_xlabel('False Positive Rate (FPR)')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].set_title('ROC Curve')
axes[0].legend(loc='lower right')
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1.02)
axes[0].grid(alpha=0.3)

# PR
prec, rec, _ = precision_recall_curve(y_test, probs_lr)
ap = average_precision_score(y_test, probs_lr)
prevalence = y_test.mean()
axes[1].plot(rec, prec, color='darkorange', lw=2, label=f'Logistic Reg. (AP = {ap:.3f})')
axes[1].axhline(prevalence, color='gray', linestyle='--', lw=1, label=f'Baseline (prevalence = {prevalence:.3f})')
axes[1].fill_between(rec, prec, alpha=0.1, color='darkorange')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='upper right')
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1.02)
axes[1].grid(alpha=0.3)

plt.tight_layout()
# plt.savefig(FIG_DIR / 'fig_roc_pr_curves.png', bbox_inches='tight')
plt.show()

## 7. Calibration

A well-calibrated model means: when it says $\hat{p}=0.3$, about 30% of those cases are truly positive. We compare logistic regression (typically well-calibrated) with a random forest (often poorly calibrated).

### Random Forest

A random forest is an ensemble of decision trees. Each tree is trained on a bootstrap sample of the data, and at each split only a random subset of features is considered. Final predictions are averaged across all trees (for probabilities) or decided by majority vote (for classes).

This randomization makes random forests robust and low-variance, but it creates a **calibration problem**: the averaged vote fractions from trees tend to cluster away from 0 and 1, compressing the predicted probability range. A forest that is highly confident will rarely output $\hat{p} = 0.95$; it saturates around $\hat{p} = 0.7$ or so. The model is *discriminative* (it ranks cases well) but *miscalibrated* (the scores are not valid probabilities).

---

### Platt Scaling

Platt scaling corrects miscalibration by fitting a logistic regression on top of the model's raw scores using a held-out calibration set:

$$\hat{p}_{\text{cal}} = \sigma(a \cdot \hat{p}_{\text{raw}} + b), \quad \sigma(z) = \frac{1}{1+e^{-z}}$$

The scalar parameters $a$ and $b$ are learned to map compressed or distorted scores onto well-calibrated probabilities. It is a post-hoc fix: the underlying model is not retrained.

> **Why it matters here:** if you lower the threshold $\tau$ to improve recall (as the previous analysis suggests), the threshold has to mean something -- $\hat{p} = 0.2$ should correspond to a true 20% event rate. Miscalibrated scores make threshold selection arbitrary.

In [ ]:
# Fit a random forest for comparison
pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
])
pipe_rf.fit(X_train, y_train)
probs_rf = pipe_rf.predict_proba(X_test)[:, 1]

# Platt-scaled RF
pipe_rf_cal = CalibratedClassifierCV(pipe_rf, method='sigmoid', cv=5)
pipe_rf_cal.fit(X_train, y_train)
probs_rf_cal = pipe_rf_cal.predict_proba(X_test)[:, 1]

In [ ]:
# Figure 5: Calibration (reliability) diagram
fig, ax = plt.subplots(figsize=(6, 5.5))

for name, probs, color, marker in [
    ('Logistic Regression', probs_lr, 'royalblue', 'o'),
    ('Random Forest', probs_rf, 'forestgreen', 's'),
    ('RF + Platt Scaling', probs_rf_cal, 'darkorange', '^'),
]:
    prob_true, prob_pred = calibration_curve(y_test, probs, n_bins=10, strategy='uniform')
    ax.plot(prob_pred, prob_true, marker=marker, label=name, color=color, lw=1.5)

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration (Reliability) Diagram')
ax.legend(loc='lower right')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
plt.tight_layout()
# plt.savefig(FIG_DIR / 'fig_calibration.png', bbox_inches='tight')
plt.show()

The dashed diagonal is the target -- a perfectly calibrated model lies on it.

**Random Forest (green):** Scores above the diagonal across most of the range, meaning it *overestimates* confidence. When it predicts $\hat{p} = 0.6$, roughly 85% of those cases are actually positive -- the model is not confident enough in its high scores, a typical symptom of vote-fraction compression.

**RF + Platt Scaling (orange):** Closer to the diagonal in the low-probability region ($\hat{p} < 0.4$), but noisy and unreliable above 0.6 due to sparse bin counts at high predicted probabilities. Platt scaling helps but does not fully correct the miscalibration here.

**Logistic Regression (blue):** Reasonable calibration at low probabilities, but the curve flattens around $\hat{p} \approx 0.3$ and stays there -- reflecting the earlier finding that LR never predicts high probabilities for positives. The model's discrimination saturates, so the high-probability bins are simply empty.

> **Key takeaway:** none of the three models is well-calibrated across the full range, which is consistent with the difficult imbalance ratio. Threshold selection should be driven by the precision-recall tradeoff rather than treating $\hat{p}$ as a literal probability.

## 8. Imbalance Strategies

We compare three approaches: (1) baseline LR, (2) class-weighted LR, (3) SMOTE + LR.
All evaluated with stratified 5-fold CV to get fair comparisons.

--- 

**1. Baseline LR** -- standard logistic regression with no adjustment for imbalance. The loss function treats every sample equally, so the majority class (no-surge) dominates and the model learns to mostly predict negative.

**2. Class-weighted LR (`class_weight='balanced'`)** -- reweights the loss so that each positive sample counts proportionally more. Specifically, the weight for class $k$ is set to $\frac{n}{2 \cdot n_k}$, where $n_k$ is the count of class $k$. No new data is created; the decision boundary shifts toward catching more positives.

**3. SMOTE + LR** -- Synthetic Minority Oversampling TEchnique generates new synthetic positive samples by interpolating between existing positives in feature space, then trains on the augmented dataset. Unlike reweighting, it physically balances the training set.

> **Important:** SMOTE is applied *inside* the pipeline, so it only sees training fold data during CV -- it is never applied to validation fold data. This is the correct implementation.

---

The training set is split into 5 folds, with each fold preserving the original class ratio (stratified). The model trains on 4 folds and predicts on the held-out fold; this rotates until every sample has been a validation sample exactly once.

`cross_val_predict(..., method='predict_proba')` assembles these out-of-fold predictions into a single probability vector covering the full training set. This gives an unbiased estimate of generalization performance without ever touching the held-out test set.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Baseline logistic regression
pipe_base = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])

# Class-weighted logistic regression
pipe_weighted = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

# SMOTE + logistic regression (imblearn Pipeline)
pipe_smote = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])

# Cross-validated predictions
probs_base_cv = cross_val_predict(pipe_base, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
probs_weighted_cv = cross_val_predict(pipe_weighted, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
probs_smote_cv = cross_val_predict(pipe_smote, X_train, y_train, cv=cv, method='predict_proba')[:, 1]

In [ ]:
# Figure 6: Comparing imbalance strategies -- ROC curves
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

models_cv = [
    ('Baseline LR', probs_base_cv),
    ('Weighted LR', probs_weighted_cv),
    ('SMOTE + LR', probs_smote_cv),
]
colors = ['royalblue', 'darkorange', 'forestgreen']

# ROC
for (name, probs), color in zip(models_cv, colors):
    fpr_m, tpr_m, _ = roc_curve(y_train, probs)
    auc_m = auc(fpr_m, tpr_m)
    axes[0].plot(fpr_m, tpr_m, lw=2, label=f'{name} (AUC={auc_m:.3f})', color=color)
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('FPR')
axes[0].set_ylabel('TPR')
axes[0].set_title('ROC -- Imbalance Strategies (5-fold CV)')
axes[0].legend(loc='lower right')
axes[0].grid(alpha=0.3)

# PR
for (name, probs), color in zip(models_cv, colors):
    prec_m, rec_m, _ = precision_recall_curve(y_train, probs)
    ap_m = average_precision_score(y_train, probs)
    axes[1].plot(rec_m, prec_m, lw=2, label=f'{name} (AP={ap_m:.3f})', color=color)
axes[1].axhline(y_train.mean(), color='gray', linestyle='--', lw=1, label='Baseline')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('PR -- Imbalance Strategies (5-fold CV)')
axes[1].legend(loc='upper right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
# plt.savefig(FIG_DIR / 'fig_imbalance_strategies.png', bbox_inches='tight')plt.show()

**ROC curves (left):** All three AUCs are 0.862. The ROC curve measures rank discrimination but is insensitive to class imbalance and absolute probability values -- a high AUC means the model orders cases reasonably well, not that it is useful in practice.

**PR curves (right):** Average Precision is nearly identical (0.200, 0.193, 0.192) and only marginally above the no-skill baseline (~0.05). Precision collapses quickly as recall increases.

**At a fixed threshold ($\tau = 0.5$):** the strategies diverge. Baseline LR predicts almost no positives (F1 = 0.003, MCC = 0.016) because its scores rarely exceed 0.5. Weighted LR and SMOTE shift the predicted probabilities upward, so the default threshold becomes operative -- both reach F1 ~ 0.254 and MCC ~ 0.28. This is not better discrimination; it is a better-positioned score distribution.

**Why do the strategies not close the gap on AP and AUC?**
Reweighting and SMOTE do not improve the underlying discriminative signal in the features. All three pipelines use the same logistic regression on the same features, so the learned ranking is essentially identical.

> **Implication:** if $\tau = 0.5$ is required, weighted LR is the practical choice. For free threshold selection, all three converge. The deeper bottleneck is feature quality -- the next step should be feature engineering or a more expressive model.

In [ ]:
# Summary table
summary = []
for name, probs in models_cv:
    y_pred = (probs >= 0.5).astype(int)
    summary.append({
        'Model': name,
        'F1': f1_score(y_train, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_train, y_pred),
        'Balanced Acc': balanced_accuracy_score(y_train, y_pred),
        'AUC': auc(*roc_curve(y_train, probs)[:2]),
        'AP': average_precision_score(y_train, probs),
    })

df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))

**Baseline LR at $\tau = 0.5$** predicts almost no positives (F1 = 0.003, MCC = 0.016) -- consistent with what the probability distribution showed earlier: the model rarely exceeds 0.5.

**Weighted LR and SMOTE + LR** both achieve F1 ~ 0.254 and MCC ~ 0.28 at the same threshold. These strategies effectively shift predicted probabilities upward, so the default $\tau = 0.5$ now captures a meaningful number of positives.

> The AUC and AP are nearly identical across all three because those metrics are threshold-agnostic -- they integrate over all thresholds. The difference only surfaces at a *fixed* threshold. Weighted LR and SMOTE are not learning better features; they are moving the score distribution so that $\tau = 0.5$ lands in a more useful operating point.

**Practical implication:** if you must use $\tau = 0.5$ (e.g. for interpretability reasons), weighted LR is the clear choice. If you tune $\tau$ freely on the PR curve, all three converge to similar performance.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()
X_train_arr = X_train.values if hasattr(X_train, 'values') else X_train
y_train_arr = y_train.values if hasattr(y_train, 'values') else y_train

for i, feature in enumerate(feature_cols):
    ax = axes[i]
    bins = np.linspace(X_train_arr[:, i].min(), X_train_arr[:, i].max(), 11)
    bin_indices = np.digitize(X_train_arr[:, i], bins) - 1
    bin_indices = np.clip(bin_indices, 0, len(bins) - 2)

    mask = lambda j: bin_indices == j
    mean_prob  = [probs_base_cv[mask(j)].mean() if mask(j).any() else np.nan for j in range(len(bins) - 1)]
    actual_rate = [y_train_arr[mask(j)].mean()  if mask(j).any() else np.nan for j in range(len(bins) - 1)]
    bin_centers = (bins[:-1] + bins[1:]) / 2

    ax.plot(bin_centers, mean_prob,   marker='o', label='Mean predicted prob', color='royalblue')
    ax.plot(bin_centers, actual_rate, marker='s', label='Actual positive rate', color='firebrick')
    ax.set_xlabel(feature)
    ax.set_ylabel('Probability / Rate')
    ax.set_title(f'Residual Analysis: {feature}')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

The gap between blue (predicted) and red (actual) reveals where logistic regression misses structure.

**Good tracking** -- blue and red move together: `temp`, `atemp`, `hum`, `windspeed`. The linear model captures the trend in continuous weather features reasonably well, though `temp` and `atemp` diverge sharply at high values (the model over-predicts surge at high temperatures).

**Poor tracking** -- systematic misfit: `hr` and `mnth`. The actual positive rate shows strong nonlinear, multi-modal patterns (e.g. surge spikes at specific hours) that logistic regression cannot represent without manual feature engineering (e.g. cyclic encoding or indicator variables).

**Categorical features** (`season`, `holiday`, `workingday`, `weathersit`): sparse bins make the comparison noisy, but the model generally underestimates the positive rate on holidays and misorders seasons.

> **Conclusion:** `hr` and `mnth` show clear nonlinear structure that logistic regression is leaving on the table. This is direct evidence that a tree-based model -- which handles such patterns without manual encoding -- should improve AP and recall.

---

**Why a tree-based model may lead to better results?**

1. Splits handle multi-modal patterns natively. The `hr` actual positive rate spikes at specific hours (e.g. rush hours) and drops elsewhere. A decision tree places splits directly at those breakpoints -- no encoding required. Logistic regression needs you to manually create indicator variables or cyclic features to approximate the same effect.

2. Interactions are captured automatically. Surge probability likely depends on combinations -- e.g. high `hr` AND high `temp`. Trees learn these conjunctions as sequences of splits. Logistic regression can only model interactions if you explicitly add product terms.

3. Monotonicity is not assumed. The `hum` plot shows a non-monotonic relationship (high positive rate at low humidity, dropping in the middle, rising again). Logistic regression assigns a single coefficient to `hum`, forcing a monotone linear effect in log-odds. A tree partitions the space freely.

> In short: wherever the red curve in the residual plots is non-monotone or multi-modal, logistic regression is structurally incapable of fitting it. Trees are not -- which is exactly what those plots show for `hr`, `mnth`, and `hum`.

## 9. Cost-Sensitive Threshold Tuning

If a false negative costs 3x a false positive, the optimal threshold is
$\tau^* = c_{FP}/(c_{FP}+c_{FN}) = 1/(1+3) = 0.25$.

Not all errors are equal. A missed surge (false negative) may be more costly than a false alarm (false positive). This block makes that asymmetry explicit.

Two costs are defined:

$$C_{FP} = 1, \qquad C_{FN} = 3$$

meaning a missed surge is penalized three times more than a false alert. For each threshold $\tau \in [0.01, 0.99]$, the total expected cost is computed:

$$\text{Cost}(\tau) = C_{FP} \cdot \text{FP}(\tau) + C_{FN} \cdot \text{FN}(\tau)$$

The optimal threshold $\tau^*$ minimizes this cost -- it is found by exhaustive search over the grid. The plot shows how cost varies with $\tau$, with $\tau^*$ marked alongside the default $\tau = 0.5$ for comparison.

> **Key point:** this is the principled way to move away from $\tau = 0.5$. Rather than tuning the threshold by inspecting metrics, you encode your cost structure directly and let the data determine the optimal operating point.

In [ ]:
# Figure 7: Expected cost vs threshold
c_fp = 1
c_fn = 3

tau_range = np.linspace(0.01, 0.99, 200)
costs = []
for tau in tau_range:
    y_pred = (probs_lr >= tau).astype(int)
    fp = np.sum((y_pred == 1) & (y_test == 0))
    fn = np.sum((y_pred == 0) & (y_test == 1))
    costs.append(c_fp * fp + c_fn * fn)

costs = np.array(costs)
optimal_tau = tau_range[np.argmin(costs)]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tau_range, costs, color='royalblue', lw=2)
ax.axvline(optimal_tau, color='firebrick', linestyle='--', lw=1.5,
           label=f'Optimal $\\tau^*={optimal_tau:.2f}$')
ax.axvline(0.5, color='gray', linestyle=':', lw=1.5, label=r'Default $\tau=0.5$')
ax.set_xlabel(r'Threshold $\tau$')
ax.set_ylabel(f'Total cost ($c_{{FP}}$={c_fp}, $c_{{FN}}$={c_fn})')
ax.set_title('Cost-Sensitive Threshold Selection')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_cost_threshold.png', bbox_inches='tight')
plt.show()

print(f"Optimal threshold: {optimal_tau:.3f}")
print(f"Cost at optimal: {costs.min():.0f} vs cost at 0.5: {costs[np.argmin(np.abs(tau_range-0.5))]:.0f}")

The cost curve drops steeply from $\tau \approx 0$ (everything flagged, dominated by FP cost) to a minimum at $\tau^* = 0.20$, then flattens from $\tau \approx 0.3$ onward.

The flat region from $\tau = 0.3$ to $1.0$ reflects the earlier finding: the model rarely predicts $\hat{p} > 0.3$, so raising the threshold further stops changing predictions. The cost stabilizes because FN count saturates (all positives are missed) and FP count reaches zero.

> The cost-optimal threshold $\tau^* = 0.20$ is consistent with the manual threshold analysis from Section 6, where $\tau = 0.2$ gave the best F1/MCC balance. Here, the same conclusion is reached from first principles using the cost ratio $C_{FN}/C_{FP} = 3$.

## Summary

| Figure | Slide Topic | Key Takeaway |
|--------|------------|--------------|
| Score distribution | Scores & Thresholds | Probabilities separate classes; threshold converts to labels |
| Threshold effects | Decision Thresholds | Lower $\tau$ catches more positives at cost of precision |
| Confusion matrices | Confusion Matrix | Same model, different matrices depending on $\tau$ |
| ROC & PR curves | ROC/PR block | ROC shows ranking; PR is more honest for rare events |
| Calibration diagram | Calibration block | LR is well-calibrated; RF needs post-hoc correction |
| Imbalance strategies | Imbalance block | Weighting/SMOTE improve recall without hurting AUC |
| Cost threshold | Putting It Together | Asymmetric costs shift $\tau$ below 0.5 |

## Bonus: Tree-based model predictions

In [ ]:
from sklearn.ensemble import RandomForestClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Baseline RF
pipep_rf = Pipeline([
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
])

# Class-weighted RF
pipep_rf_weighted = Pipeline([
    ('rf', RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                   random_state=42, n_jobs=-1))
])

# SMOTE + RF
pipep_rf_smote = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
])

# Cross-validated predictions
probs_rf_cv         = cross_val_predict(pipep_rf,         X_train, y_train, cv=cv, method='predict_proba')[:, 1]
probs_rf_weighted_cv = cross_val_predict(pipep_rf_weighted, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
probs_rf_smote_cv   = cross_val_predict(pipep_rf_smote,   X_train, y_train, cv=cv, method='predict_proba')[:, 1]

**Why RF is more robust to imbalance than LR:** Random forests use Gini impurity or entropy at each split, which are less dominated by the majority class than the logistic loss. The model also aggregates many trees, which smooths out majority-class dominance.

**When `class_weight='balanced'` still helps:** If the imbalance is severe (e.g., ~5% positive rate), the splits near leaf nodes can still be majority-class dominated. `class_weight='balanced'` adjusts the impurity criterion and is low-cost -- worth including by default.

**When SMOTE is less justified:** SMOTE interpolates between existing positives in feature space. For a random forest, which already partitions feature space via splits, synthetic interpolated points add less information than they do for a linear model. SMOTE also increases training time significantly with `n_estimators=200`.

**Practical recommendation:**

| Strategy | Worth trying? |
|---|---|
| Baseline RF | Yes, always as reference |
| `class_weight='balanced'` | Yes, nearly free |
| SMOTE + RF | Lower priority; try only if balanced RF underperforms |

---

At each split, a decision tree asks: which feature and threshold best separates the data? "Best" is measured by an impurity criterion.

**Gini impurity** for a node with class proportions $p_0, p_1$:

$$G = 1 - \sum_k p_k^2 = 2p_0 p_1$$

**Entropy:**

$$H = -\sum_k p_k \log p_k$$

Both measure how mixed the classes are in a node. A pure node (all one class) scores 0; a maximally mixed node scores highest. The tree picks the split that maximally reduces impurity in the child nodes.

**Why this helps with imbalance:** unlike logistic regression's loss, which accumulates gradients across all samples equally, impurity is computed locally per node. If a subset of the data is highly predictive of the positive class -- even if that subset is small -- the tree can isolate it with a split. The majority class does not globally suppress the minority class signal the way it does in a linear model trained end-to-end.

The remaining vulnerability is at **leaf nodes**: if a leaf contains 100 majority and 2 minority samples, the predicted probability is 0.02. `class_weight='balanced'` upweights the minority contribution to the impurity calculation, making such splits more attractive earlier in the tree.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

models_cv_rf = [
    ('Baseline RF', probs_rf_cv),
    ('Weighted RF', probs_rf_weighted_cv),
    ('SMOTE + RF', probs_rf_smote_cv),
]
colors = ['royalblue', 'darkorange', 'forestgreen']

# ROC
for (name, probs), color in zip(models_cv_rf, colors):
    fpr_m, tpr_m, _ = roc_curve(y_train, probs)
    auc_m = auc(fpr_m, tpr_m)
    axes[0].plot(fpr_m, tpr_m, lw=2, label=f'{name} (AUC={auc_m:.3f})', color=color)
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('FPR')
axes[0].set_ylabel('TPR')
axes[0].set_title('ROC -- Imbalance Strategies (5-fold CV)')
axes[0].legend(loc='lower right')
axes[0].grid(alpha=0.3)

# PR
for (name, probs), color in zip(models_cv_rf, colors):
    prec_m, rec_m, _ = precision_recall_curve(y_train, probs)
    ap_m = average_precision_score(y_train, probs)
    axes[1].plot(rec_m, prec_m, lw=2, label=f'{name} (AP={ap_m:.3f})', color=color)
axes[1].axhline(y_train.mean(), color='gray', linestyle='--', lw=1, label='Baseline')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('PR -- Imbalance Strategies (5-fold CV)')
axes[1].legend(loc='lower left')
axes[1].grid(alpha=0.3)

plt.tight_layout()

The improvement over logistic regression is significant.

**ROC (left):** All three RF variants reach AUC = 0.989, up from 0.862 with LR. The curve is nearly ideal, hugging the top-left corner.

**PR (right):** AP jumps to approx. 0.88, up from approx. 0.20 with LR. This confirms that the LR bottleneck was expressiveness, not imbalance handling. The model can now maintain high precision (approx. 0.90) even at recall = 0.5.

**Imbalance strategies:** differences are negligible -- baseline, weighted, and SMOTE RF are nearly indistinguishable on both curves. This confirms the earlier prediction: RF's split-based learning is inherently more robust to imbalance than LR, and resampling adds little on top.

> **Conclusion:** the residual analysis correctly diagnosed the problem. The nonlinear, multi-modal patterns in `hr`, `mnth`, and `hum` that LR could not capture are well within RF's capacity, and the AP improvement from 0.20 to 0.88 reflects this directly.

In [ ]:
# Fit on full training set, predict on test set
pipep_rf_smote.fit(X_train, y_train)
probs_rf_smote_test = pipep_rf_smote.predict_proba(X_test)[:, 1]

thresholds = [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]
results = []
for tau in thresholds:
    y_pred = (probs_rf_smote_test >= tau).astype(int)
    results.append({
        'Threshold': tau,
        'Pred Pos': y_pred.sum(),
        'TP': int(np.sum((y_pred == 1) & (y_test == 1))),
        'FP': int(np.sum((y_pred == 1) & (y_test == 0))),
        'FN': int(np.sum((y_pred == 0) & (y_test == 1))),
        'TN': int(np.sum((y_pred == 0) & (y_test == 0))),
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, zero_division=0),
        'F1':        f1_score(y_test, y_pred, zero_division=0),
        'MCC':       matthews_corrcoef(y_test, y_pred),
    })

df_results_rf = pd.DataFrame(results)
print(df_results_rf.to_string(index=False))

# --- plots (unchanged) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
ax.plot(df_results_rf['Threshold'], df_results_rf['Precision'], 'o-', label='Precision', color='darkorange')
ax.plot(df_results_rf['Threshold'], df_results_rf['Recall'],    's-', label='Recall',    color='forestgreen')
ax.plot(df_results_rf['Threshold'], df_results_rf['F1'],        'D-', label='F1',        color='royalblue')
ax.plot(df_results_rf['Threshold'], df_results_rf['MCC'],       '^-', label='MCC',       color='purple')
ax.set_xlabel(r'Threshold $\tau$')
ax.set_ylabel('Score')
ax.set_title('Metrics vs. Decision Threshold (RF + SMOTE)')
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)

ax = axes[1]
ax.bar(df_results_rf['Threshold'].astype(str), df_results_rf['Pred Pos'], color='steelblue', alpha=0.7)
ax.axhline(y_test.sum(), color='firebrick', linestyle='--', label=f'Actual positives ({y_test.sum()})')
ax.set_xlabel(r'Threshold $\tau$')
ax.set_ylabel('Number of predicted positives')
ax.set_title('Predicted Positives at Each Threshold (RF + SMOTE)')
ax.legend()
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

The model is strong across all thresholds (a qualitative shift from the LR results).

**Precision-recall tradeoff is well-behaved:** at $\tau = 0.1$, recall = 0.97 but precision = 0.39 (2.5x more predicted positives than actual). Raising $\tau$ improves precision rapidly while recall degrades gracefully.

**Best balanced operating point:** $\tau = 0.5$ achieves F1 = 0.847 and MCC = 0.839 -- the predicted positive count (180) closely matches the actual (174), indicating well-calibrated scores at this threshold.

**High-precision regime:** at $\tau = 0.9$, precision = 0.96 but recall collapses to 0.30 -- useful only if false alarms are extremely costly.

> For the cost structure $C_{FN} = 3$, $C_{FP} = 1$ used earlier, $\tau = 0.3$ is likely optimal: it misses only 11 surges while keeping FP manageable at 56. Compare this to LR at $\tau = 0.2$ which missed 112 surges.

In [ ]:
# Summary table
summary = []
for name, probs in models_cv_rf:
    y_pred = (probs >= 0.5).astype(int)
    summary.append({
        'Model': name,
        'F1': f1_score(y_train, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_train, y_pred),
        'Balanced Acc': balanced_accuracy_score(y_train, y_pred),
        'AUC': auc(*roc_curve(y_train, probs)[:2]),
        'AP': average_precision_score(y_train, probs),
    })

df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))

All three RF variants are strong, but SMOTE + RF stands out at the fixed threshold.

**SMOTE + RF** achieves the best F1 (0.814) and MCC (0.805), and the highest balanced accuracy (0.893) -- meaning it catches surges most effectively without sacrificing too many true negatives. This contrasts with the PR/ROC curves where SMOTE ranked last (AP = 0.872): SMOTE does not improve overall ranking quality, but it shifts the score distribution so that $\tau = 0.5$ lands at a better operating point.

**Weighted RF** is a close second (F1 = 0.773, MCC = 0.770) with negligible cost.

**Baseline RF** still performs well (F1 = 0.760) -- a massive improvement over any LR variant (best LR F1 was 0.254 at $\tau = 0.5$).

> If the threshold is fixed at 0.5, prefer SMOTE + RF. If the threshold is tuned freely on the PR curve, weighted RF or baseline RF are sufficient and cheaper to train.

In [ ]:
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

# --- 1. Fit on full training set for inspection ---
pipep_rf.fit(X_train, y_train)
rf_model_ins = pipep_rf.named_steps['rf']

# --- 2. Impurity-based feature importance ---
fig, ax = plt.subplots(figsize=(8, 5))
importances = pd.Series(rf_model_ins.feature_importances_, index=feature_cols).sort_values()
importances.plot.barh(ax=ax, color='royalblue')
ax.set_xlabel('Mean decrease in impurity')
ax.set_title('RF Feature Importance (impurity-based)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- 3. Permutation importance (on test set, scored by AP) ---
perm = permutation_importance(
    pipep_rf, X_test, y_test,
    scoring='average_precision', n_repeats=10, random_state=42, n_jobs=-1
)
perm_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm.importances_mean,
    'importance_std':  perm.importances_std
}).sort_values('importance_mean')

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(perm_df['feature'], perm_df['importance_mean'],
        xerr=perm_df['importance_std'], color='firebrick', alpha=0.8)
ax.set_xlabel('Mean decrease in AP')
ax.set_title('RF Permutation Importance')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- 4. Partial dependence plots for key features ---
key_features = ['hr', 'mnth', 'hum', 'temp']
fig, axes = plt.subplots(1, len(key_features), figsize=(16, 4))
PartialDependenceDisplay.from_estimator(
    pipep_rf, X_train, features=key_features,
    feature_names=feature_cols, ax=axes,
    pd_line_kw={'color': 'royalblue', 'lw': 2},
    percentiles=(0.05, 0.95)
)
fig.suptitle('Partial Dependence Plots -- Random Forest', y=1.02)
plt.tight_layout()
plt.show()

# --- 5. Binned residual plots (same as LR, for direct comparison) ---
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()
X_train_arr = X_train.values if hasattr(X_train, 'values') else X_train
y_train_arr = y_train.values if hasattr(y_train, 'values') else y_train

for i, feature in enumerate(feature_cols):
    ax = axes[i]
    bins = np.linspace(X_train_arr[:, i].min(), X_train_arr[:, i].max(), 11)
    bin_indices = np.clip(np.digitize(X_train_arr[:, i], bins) - 1, 0, len(bins) - 2)
    m = lambda j: bin_indices == j
    mean_prob   = [probs_rf_cv[m(j)].mean()   if m(j).any() else np.nan for j in range(len(bins)-1)]
    actual_rate = [y_train_arr[m(j)].mean()   if m(j).any() else np.nan for j in range(len(bins)-1)]
    bin_centers = (bins[:-1] + bins[1:]) / 2
    ax.plot(bin_centers, mean_prob,   marker='o', color='royalblue', label='Mean predicted prob')
    ax.plot(bin_centers, actual_rate, marker='s', color='firebrick', label='Actual positive rate')
    ax.set_xlabel(feature)
    ax.set_ylabel('Probability / Rate')
    ax.set_title(f'Residual Analysis: {feature}')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Feature importance (impurity-based):** `hr` dominates with 0.35, followed distantly by `hum` (0.11) and `atemp`/`temp` (~0.08). `holiday` is near zero. This confirms `hr` as the primary driver of surge prediction.

**Permutation importance:** `hr` remains dominant (0.70 drop in AP when shuffled), with `yr` second (0.40) -- notably higher than its impurity ranking. This suggests `yr` captures a trend (e.g. growing ridership over time) that is genuinely predictive but spread across many splits, making it undervalued by impurity. `windspeed` and `holiday` drop to near zero, confirming they add little predictive signal.

**Partial dependence plots:** `hr` shows two sharp peaks at ~8h and ~17h (rush hours), a pattern logistic regression could not capture. `mnth` has a mild upward trend, confirming its impurity importance was overstated. `hum` shows a monotone decrease -- high humidity suppresses surge probability. `temp` shows a U-shaped effect with surge probability rising above 0.6.

**Residual analysis (RF vs. LR):** blue and red curves are tightly aligned across nearly all features -- a clear improvement over LR. Remaining gaps are small and concentrated in sparse bins (`holiday`, `workingday`), where sample size limits reliability rather than model expressiveness.

> The diagnostics are mutually consistent: `hr` and `yr` drive the model, the nonlinear patterns flagged in the LR residuals are now captured, and no systematic misfit remains.